# Pharmaceutical Document RAG

A high-performance pharmaceutical document question-answering system built on `RAGPipeline`.

---

| Component | Details |
|---|---|
| **Embedding** | `sentence-transformers/all-MiniLM-L6-v2` (GPU-accelerated) |
| **Chunking** | Fixed-size (512 tokens, 50 overlap) |
| **Retrieval** | Hybrid — Vector + BM25, reciprocal rerank (top-3) |
| **LLM** | Mistral 7B Instruct (local, 4K context) |
| **OCR** | Tesseract (parallel, 200 DPI) |
| **UI** | Gradio |
| **Performance** | Index persistence + batched classification |

## 1. Install Dependencies

In [5]:
%pip install -q pymupdf
%pip install -q llama-index llama-index-core
%pip install -q llama-index-embeddings-huggingface
%pip install -q llama-index-llms-llama-cpp
%pip install -q llama-index-retrievers-bm25
%pip install -q llama-cpp-python
%pip install -q sentence-transformers huggingface-hub torch
%pip install -q pytesseract pillow
%pip install -q "gradio>=6.9.0" --upgrade
%pip install -q "nest-asyncio>=1.6.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [6]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

# Ensure the src directory is on the path so rag can be imported
sys.path.insert(0, str(Path("..").resolve() / "src"))

from rag import RAGPipeline
import gradio as gr

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 3. Initialize RAG Pipeline

Loads the GGUF model directly from disk — no download needed.

> **GPU offload:** `n_gpu_layers=-1` offloads all layers to GPU (CUDA). Set `n_gpu_layers=0` to run on CPU only.

> **Index persistence:** `persist_dir="./storage"` saves the index to disk after building, enabling instant loading on subsequent runs.

In [7]:
MODEL_PATH = r"C:\LLM Models\Mistral\mistral-7b-instruct-v0.2.Q4_K_M.gguf"

rag = RAGPipeline(model_path=MODEL_PATH, persist_dir="./storage")
print("RAGPipeline initialized.")

llama_context: n_ctx_per_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://hu

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HT

RAGPipeline initialized.


## 4. Gradio UI

Upload a pharmaceutical PDF, click **Build Pipeline** to index it, then ask questions in the chat. Each answer includes source citations and per-chunk confidence scores.

**Performance optimizations:**
- GPU-accelerated embeddings for 5-10x faster indexing
- Fixed-size chunking (20-50x faster than semantic)
- Index persistence (instant loading after first build)
- Parallel OCR processing (3-5x faster for scanned pages)
- Reduced LLM context window for faster inference

In [ ]:
_pipeline_ready = rag._query_engine is not None  # True if index was auto-loaded from persist_dir

# ── Human-readable label maps ──────────────────────────────────────────────────
_PHARMA_LABELS = {
    "cover_letter": "Cover Letter",
    "certificate_of_quality": "Certificate of Quality",
    "packaging_specification": "Packaging Specification",
    "bse_tse_declaration": "BSE / TSE Declaration",
    "material_description": "Material Description",
    "supplier_qualification": "Supplier Qualification",
    "chain_of_custody": "Chain of Custody",
    "unknown": "Unclassified",
    "unclassified": "Unclassified",
}


def _scan_label(ratio: float) -> str:
    if ratio == 0:
        return "Digital"
    if ratio < 0.5:
        return f"Mostly digital ({ratio:.0%} scanned)"
    if ratio < 1.0:
        return f"Mostly scanned ({ratio:.0%} scanned)"
    return "Fully scanned (OCR)"


def _format_document_panel(details: list) -> str:
    """Render per-file document details as user-friendly markdown cards."""
    if not details:
        return "*No documents indexed yet.*"

    lines = [f"**{len(details)} document(s) in index**\n"]
    for doc in details:
        fname = doc["file_name"]
        pages = doc["total_pages"]
        chunks = doc["total_chunks"]
        ocr = doc["has_ocr"]
        scan_ratio = doc["scan_ratio"]
        pharma_types = doc["pharma_doc_types"]

        lines.append(f"### {fname}\n")
        lines.append("| Property | Value |")
        lines.append("|---|---|")
        lines.append(f"| Pages indexed | {pages} |")
        lines.append(f"| Text chunks | {chunks} |")
        lines.append(f"| Format | {_scan_label(scan_ratio)} |")
        lines.append(f"| OCR applied | {'Yes' if ocr else 'No'} |")

        if pharma_types:
            sorted_types = sorted(pharma_types.items(), key=lambda x: -x[1])
            type_str = ", ".join(
                f"{_PHARMA_LABELS.get(k, k)} ({v}p)" for k, v in sorted_types
            )
            lines.append(f"| Document categories | {type_str} |")
        else:
            lines.append("| Document categories | *(no categories detected)* |")

        lines.append("\n---\n")

    return "\n".join(lines)


def _format_file_list(paths: list) -> str:
    if not paths:
        return "*No files queued.*"
    items = "\n".join(f"- `{os.path.basename(p)}`" for p in paths)
    return f"**{len(paths)} file(s) queued for indexing:**\n\n{items}"


def build_pipeline(pdf_files, accumulated_files):
    """Index uploaded PDFs with automatic document classification, merging with previously accumulated files."""
    global _pipeline_ready

    if pdf_files is None:
        pdf_files = []
    if not isinstance(pdf_files, list):
        pdf_files = [pdf_files]
    new_paths = [f.name if hasattr(f, "name") else f for f in pdf_files if f is not None]

    existing_names = {os.path.basename(p) for p in accumulated_files}
    for path in new_paths:
        if os.path.basename(path) not in existing_names:
            accumulated_files = accumulated_files + [path]
            existing_names.add(os.path.basename(path))

    if not accumulated_files:
        return (
            "No files uploaded.",
            "*Upload one or more PDFs and click **Build Index**.*",
            "*No documents indexed yet.*",
            accumulated_files,
            _format_file_list(accumulated_files),
        )

    try:
        _pipeline_ready = False
        num_files = len(accumulated_files)

        progress_updates = []

        def progress_callback(current, total, filename):
            progress_updates.append(f"✓ Loaded {current}/{total}: {filename}")

        if num_files == 1:
            rag.build(accumulated_files[0], classify_docs=True)
            file_label = os.path.basename(accumulated_files[0])
        else:
            rag.build_from_multiple_pdfs(
                accumulated_files,
                classify_docs=True,
                progress_callback=progress_callback,
            )
            file_label = f"{num_files} files"

        _pipeline_ready = True
        status = f"Ready — {file_label} (classified)"

        stats = rag.get_stats()
        stats_md = (
            f"| Stat | Value |\n"
            f"|---|---|\n"
            f"| Files | {stats.get('total_files', 1)} |\n"
            f"| Pages | {stats['total_pages']} |\n"
            f"| Chunks | {stats['total_chunks']} |\n"
        )
        if stats["classified"] and stats["doc_type_counts"]:
            type_rows = "\n".join(
                f"| &nbsp;&nbsp;`{_PHARMA_LABELS.get(dt, dt)}` | {count} pages |"
                for dt, count in sorted(stats["doc_type_counts"].items())
            )
            stats_md += f"| **Document categories** | |\n{type_rows}\n"
        else:
            stats_md += "| Document categories | *(no categories detected)* |\n"

        if progress_updates:
            stats_md += "\n\n**Processing log:**\n\n" + "\n".join(f"- {p}" for p in progress_updates)

        details = rag.get_document_details()
        docs_md = _format_document_panel(details)

        return status, stats_md, docs_md, accumulated_files, _format_file_list(accumulated_files)

    except Exception as exc:
        import traceback
        error_details = traceback.format_exc()
        return (
            f"Error: {exc}",
            f"```\n{error_details}\n```",
            "*Error building document index.*",
            accumulated_files,
            _format_file_list(accumulated_files),
        )


def clear_files():
    """Reset the accumulated file list and pipeline state."""
    global _pipeline_ready
    _pipeline_ready = False
    return (
        [],
        "No document loaded.",
        "*Stats will appear here after building the index.*",
        "*No documents indexed yet.*",
        "*No files queued.*",
    )


def ask(question, history, classify, expand, num_expansions, top_k, filter_doc_type):
    """Stream a RAG answer token-by-token; sources appear after retrieval."""
    if not question.strip():
        yield history, "", "*Ask a question above.*"
        return

    if not _pipeline_ready:
        yield history + [
            {"role": "user", "content": question},
            {"role": "assistant", "content": "Please upload and build a document index first."},
        ], "", "*No sources — pipeline not ready.*"
        return

    original_top_k = rag.similarity_top_k
    rag.similarity_top_k = top_k

    streaming_history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": ""},
    ]
    yield streaming_history, "", "*Retrieving sources…*"

    final_payload = None
    try:
        for chunk in rag.stream_query_with_sources(
            question,
            classify=classify,
            expand=expand,
            num_expansions=num_expansions,
        ):
            if chunk["token"] is not None:
                streaming_history[-1]["content"] += chunk["token"]
                yield streaming_history, "", "*Retrieving sources…*"
            else:
                final_payload = chunk
    except Exception as exc:
        rag.similarity_top_k = original_top_k
        yield history + [
            {"role": "user", "content": question},
            {"role": "assistant", "content": f"Error: {exc}"},
        ], "", "*Error during retrieval.*"
        return

    rag.similarity_top_k = original_top_k

    if final_payload is None:
        yield streaming_history, "", "*No response received.*"
        return

    # Guard against empty LLM output (e.g. first-call warm-up on llama.cpp)
    if not streaming_history[-1]["content"].strip():
        streaming_history[-1]["content"] = (
            "_The model returned an empty response. "
            "This can happen on the first inference call — please ask again._"
        )
        yield streaming_history, "", "*No answer generated — try again.*"
        return

    sources = final_payload["sources"]
    query_category = final_payload["query_category"]

    if filter_doc_type and filter_doc_type != "all":
        sources = [src for src in sources if src.get("pharma_doc_type") == filter_doc_type]
        if not sources:
            yield streaming_history, "", f"*No sources found for document type: {filter_doc_type}*"
            return

    sources_md = f"**{len(sources)} chunk(s) retrieved**"
    if query_category:
        sources_md += f" &nbsp;·&nbsp; Query classified as: **`{_PHARMA_LABELS.get(query_category, query_category)}`**"
    if filter_doc_type and filter_doc_type != "all":
        sources_md += f" &nbsp;·&nbsp; Filtered to: **`{_PHARMA_LABELS.get(filter_doc_type, filter_doc_type)}`**"
    sources_md += "\n\n---\n\n"

    for i, src in enumerate(sources, 1):
        confidence_str = f"{src['score']:.1f}%" if src['score'] is not None else 'N/A'
        pharma_label = _PHARMA_LABELS.get(src.get('pharma_doc_type', 'unknown'), 'Unclassified')
        scan_label = _scan_label(1.0 if src['doc_type'] == 'scanned' else 0.0)
        sources_md += (
            f"**Source {i}** &nbsp;·&nbsp; "
            f"`{src['file']}` &nbsp;·&nbsp; "
            f"Page **{src['page']}** &nbsp;·&nbsp; "
            f"Confidence: **{confidence_str}** &nbsp;·&nbsp; "
            f"{scan_label} &nbsp;·&nbsp; "
            f"**{pharma_label}**\n\n"
            f"> {src['text'][:300]}{'...' if len(src['text']) > 300 else ''}\n\n"
            f"---\n\n"
        )

    yield streaming_history, "", sources_md

# ── Pfizer-branded Gradio UI ───────────────────────────────────────────────────

PFIZER_CSS = """
/* Minimal, high-contrast Pfizer styling */
:root {
    --pf-bg: #f7f8fb;
    --pf-surface: #ffffff;
    --pf-surface-alt: #f3f6fa;
    --pf-border: #d9e1ea;
    --pf-text: #172132;
    --pf-muted: #526073;
    --pf-primary: #005eb8;
    --pf-primary-strong: #004b92;
    --pf-danger: #d4334f;
    --pf-radius: 10px;
}

.gradio-container {
    font-family: 'Source Sans 3', 'IBM Plex Sans', 'Segoe UI', sans-serif !important;
    background: var(--pf-bg) !important;
    max-width: 1280px !important;
    color: var(--pf-text) !important;
}

/* Header: simplified from gradient to clean card */
.pf-header {
    background: var(--pf-surface);
    border: 1px solid var(--pf-border);
    border-left: 5px solid var(--pf-primary);
    border-radius: var(--pf-radius);
    padding: 16px 20px;
    margin-bottom: 16px;
    display: flex;
    align-items: center;
    justify-content: space-between;
}
.pf-header-title { color: var(--pf-text); font-size: 1.25rem; font-weight: 700; margin: 0; line-height: 1.2; }
.pf-header-sub { color: var(--pf-muted); font-size: 0.85rem; margin: 3px 0 0; }
.pf-badge {
    background: var(--pf-surface-alt);
    color: var(--pf-primary-strong);
    border: 1px solid var(--pf-border);
    padding: 4px 10px;
    border-radius: 999px;
    font-size: 0.7rem;
    font-weight: 700;
    letter-spacing: 0.08em;
    text-transform: uppercase;
}

.pf-section {
    font-size: 0.68rem;
    font-weight: 700;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    color: var(--pf-muted) !important;
    margin: 0 0 10px;
    padding-bottom: 6px;
    border-bottom: 1px solid var(--pf-border);
}
.pf-divider { border: none; border-top: 1px solid var(--pf-border); margin: 16px 0; }

/* Shared readable surfaces */
.gradio-container .block,
.gradio-container .gr-box,
.gradio-container .gr-panel,
.gradio-container .gr-accordion,
.gradio-container details {
    background: var(--pf-surface) !important;
    border: 1px solid var(--pf-border) !important;
    border-radius: var(--pf-radius) !important;
    color: var(--pf-text) !important;
    box-shadow: none !important;
}

button.lg.primary, button.primary {
    background: var(--pf-primary) !important;
    border: 1px solid var(--pf-primary) !important;
    color: #ffffff !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
}
button.lg.primary:hover, button.primary:hover {
    background: var(--pf-primary-strong) !important;
    border-color: var(--pf-primary-strong) !important;
}
button.stop, button.secondary {
    background: var(--pf-surface) !important;
    border: 1px solid var(--pf-border) !important;
    color: var(--pf-text) !important;
    border-radius: 8px !important;
}
button.stop:hover { border-color: var(--pf-danger) !important; color: var(--pf-danger) !important; }

textarea, input[type="text"], .gradio-container select {
    background: var(--pf-surface) !important;
    color: var(--pf-text) !important;
    border: 1px solid var(--pf-border) !important;
    border-radius: 8px !important;
}
textarea:focus, input[type="text"]:focus, .gradio-container select:focus {
    border-color: var(--pf-primary) !important;
    box-shadow: 0 0 0 2px rgba(0,94,184,0.16) !important;
}
textarea::placeholder, input[type="text"]::placeholder { color: var(--pf-muted) !important; }

.gradio-container label,
.gradio-container label span,
.gradio-container .label-wrap,
.gradio-container .label-wrap span,
.gradio-container .gr-accordion > .label-wrap,
.gradio-container details > summary {
    color: var(--pf-text) !important;
    background: transparent !important;
}
.gradio-container .info,
.gradio-container span.info { color: var(--pf-muted) !important; }

#pf-chatbot {
    border: 1px solid var(--pf-border) !important;
    border-radius: var(--pf-radius) !important;
    background: var(--pf-surface) !important;
}
#pf-chatbot .message-row.user-row .message,
#pf-chatbot .user-row .message {
    background: var(--pf-primary) !important;
    color: #ffffff !important;
    border-radius: 12px 12px 2px 12px !important;
}
#pf-chatbot .message-row.user-row .prose,
#pf-chatbot .message-row.user-row .prose p,
#pf-chatbot .message-row.user-row .prose li,
#pf-chatbot .user-row .prose,
#pf-chatbot .user-row .prose p,
#pf-chatbot .user-row .prose li { color: #ffffff !important; }

#pf-chatbot .message-row.bot-row .message,
#pf-chatbot .bot-row .message {
    background: #121212 !important;
    border: 1px solid #202020 !important;
    color: #ffffff !important;
    border-radius: 12px 12px 12px 2px !important;
}
#pf-chatbot .message-row.bot-row .prose,
#pf-chatbot .message-row.bot-row .prose p,
#pf-chatbot .message-row.bot-row .prose li,
#pf-chatbot .bot-row .prose,
#pf-chatbot .bot-row .prose p,
#pf-chatbot .bot-row .prose li { color: #ffffff !important; }

input[type="checkbox"]:checked, input[type="range"] { accent-color: var(--pf-primary) !important; }

ul[role="listbox"] {
    background: var(--pf-surface) !important;
    border: 1px solid var(--pf-border) !important;
}
ul[role="listbox"] li { color: var(--pf-text) !important; background: var(--pf-surface) !important; }
ul[role="listbox"] li:hover, ul[role="listbox"] li[aria-selected="true"] {
    background: var(--pf-surface-alt) !important;
    color: var(--pf-text) !important;
}

.status-box textarea {
    color: var(--pf-text) !important;
    background: var(--pf-surface-alt) !important;
    border-color: var(--pf-border) !important;
}

.gradio-container .prose,
.gradio-container .prose p,
.gradio-container .prose li,
.gradio-container .prose strong,
.gradio-container .prose em,
.gradio-container .prose h1,
.gradio-container .prose h2,
.gradio-container .prose h3,
.gradio-container .prose h4 { color: var(--pf-text) !important; }

/* Document detail markdown tables: black theme */
.gradio-container .prose table {
    border-collapse: collapse !important;
    width: 100% !important;
}
.gradio-container .prose table th,
.gradio-container .prose table td {
    background: #000000 !important;
    color: #ffffff !important;
    border: 1px solid #2a2a2a !important;
}
.gradio-container .prose table tr:hover td {
    background: #111111 !important;
    color: #ffffff !important;
}
"""

_PFIZER_THEME = gr.themes.Base(
    primary_hue=gr.themes.colors.blue,
    neutral_hue=gr.themes.colors.gray,
    font=[gr.themes.GoogleFont("Source Sans 3"), gr.themes.GoogleFont("IBM Plex Sans"), "Segoe UI", "sans-serif"],
).set(
    body_background_fill="#f7f8fb",
    body_text_color="#172132",
    body_text_color_subdued="#526073",
    block_background_fill="#ffffff",
    block_border_color="#d9e1ea",
    block_border_width="1px",
    block_radius="10px",
    block_shadow="0 0 0 rgba(0,0,0,0)",
    block_title_text_color="#172132",
    block_label_text_color="#172132",
    input_background_fill="#ffffff",
    input_border_color="#d9e1ea",
    input_border_color_focus="#005eb8",
    input_placeholder_color="#526073",
    button_primary_background_fill="#005eb8",
    button_primary_background_fill_hover="#004b92",
    button_primary_text_color="#ffffff",
    button_secondary_background_fill="#ffffff",
    button_secondary_border_color="#d9e1ea",
    button_secondary_text_color="#172132",
    checkbox_label_text_color="#172132",
    checkbox_label_text_color_selected="#172132",
    checkbox_background_color_selected="#005eb8",
)

with gr.Blocks(title="Pfizer | Pharmaceutical Document QA") as demo:

    # ── Header ──────────────────────────────────────────────────────────────────
    gr.HTML("""
    <div class="pf-header">
        <div>
            <p class="pf-header-title">Pharmaceutical Document QA</p>
            <p class="pf-header-sub">Upload pharmaceutical PDFs and ask questions — powered by hybrid RAG retrieval with automatic document classification</p>
        </div>
        <span class="pf-badge">RAG System</span>
    </div>
    """)

    accumulated_files_state = gr.State([])

    # ── Main layout ─────────────────────────────────────────────────────────────
    with gr.Row(equal_height=False):

        # Left column — document management
        with gr.Column(scale=2, min_width=300):
            gr.HTML('<p class="pf-section">Documents</p>')

            pdf_input = gr.File(
                label="Upload PDF(s)",
                file_types=[".pdf"],
                file_count="multiple",
            )

            with gr.Row():
                build_btn = gr.Button("Build Index", variant="primary", size="lg", scale=3)
                clear_btn = gr.Button("Clear", variant="stop", size="sm", scale=1)

            status_box = gr.Textbox(
                label="Status",
                value="No document loaded." if not _pipeline_ready else "Ready — index loaded from disk.",
                interactive=False,
                max_lines=2,
                elem_classes=["status-box"],
            )

            gr.HTML('<hr class="pf-divider"/>')

            with gr.Accordion("Queued Files", open=False):
                queued_files_display = gr.Markdown("*No files queued.*")

            with gr.Accordion("Document Details", open=True):
                docs_display = gr.Markdown("*No documents indexed yet.*")

            with gr.Accordion("Index Summary", open=False):
                stats_display = gr.Markdown("*Stats will appear here after building the index.*")

        # Right column — chat interface
        with gr.Column(scale=3, min_width=420):
            gr.HTML('<p class="pf-section">Query</p>')

            chatbot = gr.Chatbot(
                height=440,
                show_label=False,
                elem_id="pf-chatbot",
                placeholder=(
                    "<div style='text-align:center;padding:40px 20px'>"
                    "<p style='font-size:1.1rem;font-weight:600;margin-bottom:6px;color:#3D4552'>No conversation yet</p>"
                    "<p style='font-size:0.85rem;color:#9BA4B0'>Build the index and ask a question to get started.</p>"
                    "</div>"
                ),
            )

            with gr.Row():
                question_input = gr.Textbox(
                    placeholder="e.g. What are the storage conditions for this batch?",
                    show_label=False,
                    scale=5,
                    lines=1,
                    max_lines=4,
                    container=False,
                )
                ask_btn = gr.Button("Ask", variant="primary", scale=1, min_width=72)

            with gr.Accordion("Sources & Evidence", open=True):
                sources_display = gr.Markdown(
                    "*Sources and confidence scores will appear here after asking a question.*"
                )

    # ── Advanced settings ────────────────────────────────────────────────────────
    with gr.Accordion("Advanced Settings", open=False):
        with gr.Row():
            with gr.Column():
                gr.Markdown("**Retrieval**")
                top_k = gr.Slider(
                    minimum=1, maximum=20, value=5, step=1,
                    label="Chunks to retrieve (top-k)",
                    info="Higher values provide more context but may slow inference.",
                )
                filter_doc_type = gr.Dropdown(
                    choices=[
                        ("All document types", "all"),
                        ("Cover Letter", "cover_letter"),
                        ("Certificate of Quality", "certificate_of_quality"),
                        ("Packaging Specification", "packaging_specification"),
                        ("BSE / TSE Declaration", "bse_tse_declaration"),
                        ("Material Description", "material_description"),
                        ("Supplier Qualification", "supplier_qualification"),
                        ("Chain of Custody", "chain_of_custody"),
                        ("Unclassified", "unknown"),
                    ],
                    value="all",
                    label="Filter by document type",
                    info="Requires classification to be enabled during indexing.",
                )
            with gr.Column():
                gr.Markdown("**Query Processing**")
                classify_query_toggle = gr.Checkbox(
                    label="Auto-classify query",
                    value=False,
                    info="Uses the LLM to detect the most relevant pharma document category.",
                )
                expand_query_toggle = gr.Checkbox(
                    label="Query expansion",
                    value=False,
                    info="Generates alternative phrasings to improve retrieval recall.",
                )
                num_expansions = gr.Slider(
                    minimum=1, maximum=5, value=3, step=1,
                    label="Number of expansions",
                    info="Only active when query expansion is enabled.",
                )

    # ── Event wiring ─────────────────────────────────────────────────────────────
    build_btn.click(
        build_pipeline,
        inputs=[pdf_input, accumulated_files_state],
        outputs=[status_box, stats_display, docs_display, accumulated_files_state, queued_files_display],
    )

    clear_btn.click(
        clear_files,
        inputs=[],
        outputs=[accumulated_files_state, status_box, stats_display, docs_display, queued_files_display],
    )

    ask_btn.click(
        ask,
        inputs=[question_input, chatbot, classify_query_toggle, expand_query_toggle, num_expansions, top_k, filter_doc_type],
        outputs=[chatbot, question_input, sources_display],
    )

    question_input.submit(
        ask,
        inputs=[question_input, chatbot, classify_query_toggle, expand_query_toggle, num_expansions, top_k, filter_doc_type],
        outputs=[chatbot, question_input, sources_display],
    )

demo.launch(share=False, css=PFIZER_CSS, theme=_PFIZER_THEME)

IndentationError: unindent does not match any outer indentation level (<string>, line 596)